# Phase 2 - Data Cleaning

- Inspect for missing values and inconsistencies
- Remove duplicates where necessary
- Inspect whether rating values are valid(0.5 - 5)
- Prepare user-movie rating data for modeling
- Create training and testing datasets

In [15]:
import pandas as pd
import numpy as np

ratings = pd.read_csv('ratings.csv')
movies  = pd.read_csv('movies.csv')
tags    = pd.read_csv('tags.csv')
links   = pd.read_csv('links.csv')

print(movies.head())

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  


In [5]:
# --- Missing values per table ---
for name, df in [("ratings", ratings), ("movies", movies), ("tags", tags), ("links", links)]:
    print(f"\n{name}: missing values")
    print(df.isna().sum()[df.isna().sum() > 0].to_string() or "  none")

# --- Duplicate checks ---
print("\nDuplicate (userId, movieId) ratings:", ratings.duplicated(["userId", "movieId"]).sum())
print("Duplicate movieIds in movies:", movies["movieId"].duplicated().sum())

# --- Are rating values valid? (expected: 0.5 to 5.0 in 0.5 steps) ---
valid = np.arange(0.5, 5.01, 0.5)
print("\nUnique rating values:", sorted(ratings["rating"].unique()))
print("Invalid ratings:", (~ratings["rating"].isin(valid)).sum())

# --- Referential integrity: do all IDs in ratings/tags exist in movies? ---
print("\nRatings with movieId not in movies:", (~ratings["movieId"].isin(movies["movieId"])).sum())
print("Tags with movieId not in movies:",    (~tags["movieId"].isin(movies["movieId"])).sum())

# --- Movies in catalog that nobody rated (only tagged) ---
unrated = set(movies["movieId"]) - set(ratings["movieId"])
print(f"Catalog movies with zero ratings: {len(unrated)}")

# --- Movies with no genre listed ---
no_genre = movies[movies["genres"] == "(no genres listed)"]
print(f"Movies with '(no genres listed)': {len(no_genre)}")

# --- Titles that appear more than once (possible duplicate entries) ---
dup_titles = movies[movies["title"].duplicated(keep=False)].sort_values("title")
print(f"Movies sharing a title with another movieId: {len(dup_titles)}")
display(dup_titles.head(10))

# --- Missing years in titles (checked after extraction below) ---


ratings: missing values
Series([], )

movies: missing values
Series([], )

tags: missing values
Series([], )

links: missing values
tmdbId    8

Duplicate (userId, movieId) ratings: 0
Duplicate movieIds in movies: 0

Unique rating values: [np.float64(0.5), np.float64(1.0), np.float64(1.5), np.float64(2.0), np.float64(2.5), np.float64(3.0), np.float64(3.5), np.float64(4.0), np.float64(4.5), np.float64(5.0)]
Invalid ratings: 0

Ratings with movieId not in movies: 0
Tags with movieId not in movies: 0
Catalog movies with zero ratings: 18
Movies with '(no genres listed)': 34
Movies sharing a title with another movieId: 10


,movieId,title,genres
4169,6003,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Thriller
9106,144606,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Romance|Thriller
650,838,Emma (1996),Comedy|Drama|Romance
5601,26958,Emma (1996),Romance
5854,32600,Eros (2004),Drama
9135,147002,Eros (2004),Drama|Romance
2141,2851,Saturn 3 (1980),Adventure|Sci-Fi|Thriller
9468,168358,Saturn 3 (1980),Sci-Fi|Thriller
5931,34048,War of the Worlds (2005),Action|Adventure|Sci-Fi|Thriller
6932,64997,War of the Worlds (2005),Action|Sci-Fi


No missing values in ratings, movies or tags.
- No duplicate (userId, movieId) rating pairs and no duplicate movieIds.
- All ratings are valid (0.5 to 5.0 in half-star steps).
- No orphan IDs: every movieId in ratings, tags and links exists in movies.


In [6]:
#pull the release year out of the title
movies_clean = movies.copy()
movies_clean["title"] = movies_clean["title"].str.strip()

movies_clean["year"] = (movies_clean["title"]
                        .str.extract(r"\((\d{4})\)\s*$")[0]
                        .astype("Int64"))
movies_clean["title_clean"] = (movies_clean["title"]
                               .str.replace(r"\s*\(\d{4}\)\s*$", "", regex=True)
                               .str.strip())

print("Movies with no year:", movies_clean["year"].isna().sum())
display(movies_clean[movies_clean["year"].isna()][["movieId", "title"]])
display(movies_clean[["title", "title_clean", "year"]].head())

Movies with no year: 13


,movieId,title
6059,40697,Babylon 5
9031,140956,Ready Player One
9091,143410,Hyena Road
9138,147250,The Adventures of Sherlock Holmes and Doctor W...
9179,149334,Nocturnal Animals
9259,156605,Paterson
9367,162414,Moonlight
9448,167570,The OA
9514,171495,Cosmos
9515,171631,Maria Bamford: Old Baby


,title,title_clean,year
0,Toy Story (1995),Toy Story,1995
1,Jumanji (1995),Jumanji,1995
2,Grumpier Old Men (1995),Grumpier Old Men,1995
3,Waiting to Exhale (1995),Waiting to Exhale,1995
4,Father of the Bride Part II (1995),Father of the Bride Part II,1995


- Year is only an optional content feature so a missing value costs nothing

In [7]:
#catalog movies with zero ratings
rated_ids = set(ratings["movieId"])
movies_clean["has_ratings"] = movies_clean["movieId"].isin(rated_ids)

unrated = movies_clean[~movies_clean["has_ratings"]]
print("Catalog movies with zero ratings:", len(unrated))
display(unrated[["movieId", "title", "genres"]])

print("Of these, how many have tags:", unrated["movieId"].isin(tags["movieId"]).sum())

Catalog movies with zero ratings: 18


,movieId,title,genres
816,1076,"Innocents, The (1961)",Drama|Horror|Thriller
2211,2939,Niagara (1953),Drama|Thriller
2499,3338,For All Mankind (1989),Documentary
2587,3456,"Color of Paradise, The (Rang-e khoda) (1999)",Drama
3118,4194,I Know Where I'm Going! (1945),Drama|Romance|War
4037,5721,"Chosen, The (1981)",Drama
4506,6668,"Road Home, The (Wo de fu qin mu qin) (1999)",Drama|Romance
4598,6849,Scrooge (1970),Drama|Fantasy|Musical
4704,7020,Proof (1991),Comedy|Drama|Romance
5020,7792,"Parallax View, The (1974)",Thriller


Of these, how many have tags: 18


| Catalog movies with zero ratings | 18 | Flagged with `has_ratings = False`, kept in the table | CF can't score them, but all 18 have genres and tags, so a content-based fallback can recommend them |

In [8]:
#the 34 movies with no genre
movies_clean["no_genre"] = movies_clean["genres"].eq("(no genres listed)")
movies_clean["genre_list"] = movies_clean["genres"].apply(
    lambda g: [] if g == "(no genres listed)" else g.split("|"))

no_genre = movies_clean[movies_clean["no_genre"]].copy()
no_genre["n_ratings"] = (no_genre["movieId"]
                         .map(ratings["movieId"].value_counts())
                         .fillna(0).astype(int))

print("Movies with no genre:", len(no_genre))
print("Of these, unrated:", (no_genre["n_ratings"] == 0).sum())
print("Total ratings on these movies:", no_genre["n_ratings"].sum())
display(no_genre[["movieId", "title", "n_ratings"]]
        .sort_values("n_ratings", ascending=False).head(10))

Movies with no genre: 34
Of these, unrated: 0
Total ratings on these movies: 47


,movieId,title,n_ratings
8687,122896,Pirates of the Caribbean: Dead Men Tell No Tal...,7
9053,141866,Green Room (2015),3
9514,171495,Cosmos,2
9541,172591,The Godfather Trilogy: 1972-1990 (1992),2
9070,142456,The Brand New Testament (2015),2
9217,152037,Grease Live (2016),2
9426,166024,Whiplash (2013),2
8517,114335,La cravate (1957),1
8684,122888,Ben-hur (2016),1
8902,134861,Trevor Noah: African American (2013),1


| Movies with "(no genres listed)" | 34 | Flagged with `no_genre`, genre list left empty | Only 47 ratings in total (0.05%), so CF is unaffected |
it can use their tags instead

In [9]:
#the 10 movies that share a title
movies_clean["dup_title"] = movies_clean["title"].duplicated(keep=False)

counts = ratings["movieId"].value_counts()
dups = movies_clean.loc[movies_clean["dup_title"], ["movieId", "title", "genres"]].copy()
dups["n_ratings"] = dups["movieId"].map(counts).fillna(0).astype(int)
dups = dups.merge(links[["movieId", "imdbId"]], on="movieId", how="left")

display(dups.sort_values(["title", "movieId"]))

,movieId,title,genres,n_ratings,imdbId
2,6003,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Thriller,15,290538
7,144606,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Romance|Thriller,1,270288
0,838,Emma (1996),Comedy|Drama|Romance,30,116191
3,26958,Emma (1996),Romance,1,118308
4,32600,Eros (2004),Drama,1,377059
8,147002,Eros (2004),Drama|Romance,1,343663
1,2851,Saturn 3 (1980),Adventure|Sci-Fi|Thriller,4,81454
9,168358,Saturn 3 (1980),Sci-Fi|Thriller,1,79285
5,34048,War of the Worlds (2005),Action|Adventure|Sci-Fi|Thriller,50,407304
6,64997,War of the Worlds (2005),Action|Sci-Fi,2,449040


| Movies sharing a title with another movieId | 10 (5 pairs) | Flagged with `dup_title`, not merged | Every pair has a different IMDb ID, and the extra entries have only 1 to 2 ratings each |

In [10]:
#one-hot encode the genres
movies_clean["genre_list"] = movies_clean["genres"].apply(
    lambda g: [] if g == "(no genres listed)" else g.split("|"))

all_genres = sorted({g for lst in movies_clean["genre_list"] for g in lst})
genre_cols = pd.DataFrame(
    {f"genre_{g}": movies_clean["genre_list"].apply(lambda lst, g=g: int(g in lst))
     for g in all_genres})
movies_clean = pd.concat([movies_clean, genre_cols], axis=1)

print("Genres found:", len(all_genres))
print(genre_cols.sum().sort_values(ascending=False).to_string())

Genres found: 19
genre_Drama          4361
genre_Comedy         3756
genre_Thriller       1894
genre_Action         1828
genre_Romance        1596
genre_Adventure      1263
genre_Crime          1199
genre_Sci-Fi          980
genre_Horror          978
genre_Fantasy         779
genre_Children        664
genre_Animation       611
genre_Mystery         573
genre_Documentary     440
genre_War             382
genre_Musical         334
genre_Western         167
genre_IMAX            158
genre_Film-Noir        87


- The genres encoded correctly and the 'no genres listed' did not turn into a fake genre column

In [11]:
#clean the tags table
tags_clean = tags.copy()

print("Raw unique tags:", tags_clean["tag"].nunique())

tags_clean["tag"] = tags_clean["tag"].astype(str).str.lower().str.strip()
tags_clean = tags_clean[tags_clean["tag"] != ""]
tags_clean = tags_clean.drop_duplicates(subset=["userId", "movieId", "tag"]).reset_index(drop=True)

print("Clean unique tags:", tags_clean["tag"].nunique())
print("Tag rows:", len(tags), "->", len(tags_clean))
print("Movies with at least one tag:", tags_clean["movieId"].nunique(), "of", len(movies_clean))
print("Users who tagged:", tags_clean["userId"].nunique())

print("\nMost common tags:")
print(tags_clean["tag"].value_counts().head(15).to_string())

Raw unique tags: 1589
Clean unique tags: 1475
Tag rows: 3683 -> 3683
Movies with at least one tag: 1572 of 9742
Users who tagged: 58

Most common tags:
tag
in netflix queue      131
atmospheric            41
superhero              24
thought-provoking      24
surreal                24
funny                  24
disney                 23
sci-fi                 23
quirky                 22
religion               22
psychology             21
dark comedy            21
suspense               21
twist ending           20
visually appealing     20


**Tags table findings**
- **Cleaning:** lowercasing and stripping merged 1,589 raw tags into 1,475 unique tags, with no duplicate rows (all 3,683 rows kept).
- **Coverage:** only 1,572 of 9,742 movies (16%) and 58 of 610 users have tags, so they can supplement genres but not replace them.
- **Noise:** "in netflix queue" appears 131 times and describes user behavior rather than the movie, so it is removed in the next step.

In [12]:
tags_clean = tags_clean[tags_clean["tag"] != "in netflix queue"].reset_index(drop=True)
print("Tag rows after removing 'in netflix queue':", len(tags_clean))

tags_per_movie = (tags_clean.groupby("movieId")["tag"]
                  .apply(lambda s: " ".join(sorted(set(s))))
                  .rename("tag_text"))

movies_clean = movies_clean.drop(columns=["tag_text"], errors="ignore")
movies_clean = movies_clean.merge(tags_per_movie, on="movieId", how="left")
movies_clean["tag_text"] = movies_clean["tag_text"].fillna("")
movies_clean["has_tags"] = movies_clean["tag_text"] != ""

print("Movies with usable tags:", movies_clean["has_tags"].sum(), "of", len(movies_clean))
display(movies_clean.loc[movies_clean["has_tags"], ["title", "tag_text"]].head())

Tag rows after removing 'in netflix queue': 3552
Movies with usable tags: 1463 of 9742


,title,tag_text
0,Toy Story (1995),fun pixar
1,Jumanji (1995),fantasy game magic board game robin williams
2,Grumpier Old Men (1995),moldy old
4,Father of the Bride Part II (1995),pregnancy remake
6,Sabrina (1995),remake


| Tags cover few movies | 1,463 of 9,742 movies (15%), 58 users | Used only as a supplement to genres | Too sparse to be the main content signal |
| Non-descriptive tag "in netflix queue" | 131 rows | Removed (3,683 to 3,552 rows) | Describes user behavior, not the movie; 109 movies lost their only tag |

In [13]:
#clean the links table
links_clean = links.copy()

print("Missing values:\n", links_clean.isna().sum().to_string())
print("\nDuplicate movieId:", links_clean["movieId"].duplicated().sum())
print("Duplicate imdbId:", links_clean["imdbId"].duplicated().sum())
print("links movieIds not in movies:", (~links_clean["movieId"].isin(movies["movieId"])).sum())

# tmdbId is float64 only because of the missing values, so convert to nullable integer
links_clean["tmdbId"] = links_clean["tmdbId"].astype("Int64")

display(links_clean[links_clean["tmdbId"].isna()])
print(links_clean.dtypes.to_string())

Missing values:
 movieId    0
imdbId     0
tmdbId     8

Duplicate movieId: 0
Duplicate imdbId: 0
links movieIds not in movies: 0


,movieId,imdbId,tmdbId
624,791,113610,<NA>
843,1107,102336,<NA>
2141,2851,81454,<NA>
3027,4051,56600,<NA>
5532,26587,92337,<NA>
5854,32600,377059,<NA>
6059,40697,105946,<NA>
7382,79299,874957,<NA>


movieId    int64
imdbId     int64
tmdbId     Int64


| Links with missing `tmdbId` | 8 rows | Kept as missing, converted to nullable integer | Not used for modeling, and there are no duplicate or orphan IDs |

In [14]:
#Final validation of all steps
ratings_clean = ratings.astype(
    {"userId": "int32", "movieId": "int32", "rating": "float32"}
).copy()

assert len(movies_clean) == 9742
assert movies_clean["movieId"].is_unique
assert not ratings_clean.duplicated(["userId", "movieId"]).any()
assert ratings_clean["rating"].between(0.5, 5.0).all()
assert ratings_clean.notna().all().all()
assert ratings_clean["movieId"].isin(movies_clean["movieId"]).all()
assert tags_clean["movieId"].isin(movies_clean["movieId"]).all()
assert links_clean["movieId"].isin(movies_clean["movieId"]).all()
assert movies_clean["has_ratings"].sum() == 9724
assert "in netflix queue" not in set(tags_clean["tag"])
print("All checks passed")

for name, df in [("ratings_clean", ratings_clean), ("movies_clean", movies_clean),
                 ("tags_clean", tags_clean), ("links_clean", links_clean)]:
    print(f"{name:14s} {df.shape[0]:>8,} rows x {df.shape[1]} cols")

All checks passed
ratings_clean   100,836 rows x 4 cols
movies_clean      9,742 rows x 30 cols
tags_clean        3,552 rows x 4 cols
links_clean       9,742 rows x 3 cols


In [16]:
#create the training and testing sets
import os, shutil
os.makedirs("processed", exist_ok=True)
test = ratings_clean.groupby("userId", group_keys=False).sample(frac=0.2, random_state=42)
train = ratings_clean.drop(test.index)

print("Train:", len(train), "| Test:", len(test))
print("Users in train:", train["userId"].nunique(), "| in test:", test["userId"].nunique())
print("Test movies never seen in train:", (~test["movieId"].isin(train["movieId"])).sum(), "rows")

train.to_csv("processed/train.csv", index=False)
test.to_csv("processed/test.csv", index=False)

Train: 80672 | Test: 20164
Users in train: 610 | in test: 610
Test movies never seen in train: 790 rows


**Train/test split:** 20% of each user's ratings were held out (80,672 train / 20,164 test, seed 42). All 610 users appear in both sets. 790 test rows (3.9%) involve movies absent from training, which collaborative filtering cannot score and a content-based fallback must handle.

In [17]:
#contiguous user and movie indices
user_map = {u: i for i, u in enumerate(sorted(ratings_clean["userId"].unique()))}
movie_map = {m: i for i, m in enumerate(sorted(movies_clean["movieId"].unique()))}

def add_idx(df):
    return df.assign(user_idx=df["userId"].map(user_map).astype("int32"),
                     movie_idx=df["movieId"].map(movie_map).astype("int32"))

ratings_clean, train, test = add_idx(ratings_clean), add_idx(train), add_idx(test)
movies_clean["movie_idx"] = movies_clean["movieId"].map(movie_map).astype("int32")

train.to_csv("processed/train.csv", index=False)
test.to_csv("processed/test.csv", index=False)
print(train[["userId", "user_idx", "movieId", "movie_idx"]].head())
print("user_idx range:", train["user_idx"].min(), "-", train["user_idx"].max())
print("movie_idx range:", train["movie_idx"].min(), "-", train["movie_idx"].max())

   userId  user_idx  movieId  movie_idx
0       1         0        1          0
1       1         0        3          2
2       1         0        6          5
3       1         0       47         43
4       1         0       50         46
user_idx range: 0 - 609
movie_idx range: 0 - 9741


In [18]:
#saving cleaned files
import os, shutil
os.makedirs("processed", exist_ok=True)

# CSV can't store Python lists, so save genre_list as pipe-separated text (empty = no genre)
movies_out = movies_clean.copy()
movies_out["genre_list"] = movies_out["genre_list"].apply("|".join)

ratings_clean.to_csv("processed/ratings_clean.csv", index=False)
movies_out.to_csv("processed/movies_clean.csv", index=False)
tags_clean.to_csv("processed/tags_clean.csv", index=False)
links_clean.to_csv("processed/links_clean.csv", index=False)

# Reload each file to confirm it saved correctly
for name in ["ratings_clean", "movies_clean", "tags_clean", "links_clean"]:
    df = pd.read_csv(f"processed/{name}.csv")
    print(f"{name:14s} {df.shape[0]:>8,} rows x {df.shape[1]} cols")

# Zip and download, since Colab's temporary files disappear when the session ends
shutil.make_archive("processed", "zip", "processed")
from google.colab import files
files.download("processed.zip")

ratings_clean   100,836 rows x 6 cols
movies_clean      9,742 rows x 31 cols
tags_clean        3,552 rows x 4 cols
links_clean       9,742 rows x 3 cols


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

No users or movies were dropped.* The long tail is exactly what the cold-start and diversity objectives measure.
- *No minimum-rating threshold was applied.* Any such filtering belongs in the modeling stage.
- *Flags instead of deletions.* has_ratings, no_genre, dup_title and has_tags record every problem so it can be handled later.
- *Scope:* this section covers cleaning and preprocessing, including the train/test split.